# 1. 載入PDF

## 安裝套件

```shell
pip install langchain langchain-community pypdf
```

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

# 1. 指定 PDF 路徑
pdf_path = "../data/raw/solid_principles.pdf"

# 2. 建立 Loader
loader = PyPDFLoader(pdf_path)

# 3. 載入文件（每一頁是一個 Document）
documents: list[Document] = loader.load()

# 4. 基本檢查
print(f"成功載入頁數：{len(documents)}")
print(f"Document 類型：{type(documents[0])}")
print(f"第一頁內容長度：{len(documents[0].page_content)}")
print(f"第一頁 metadata：")
for key, value in documents[0].metadata.items():
    print(f"  - {key}: {value}")

print("\n--- 第一頁內容預覽 ---")
print(documents[0].page_content[:200] + "...")

成功載入頁數：9
Document 類型：<class 'langchain_core.documents.base.Document'>
第一頁內容長度：788
第一頁 metadata：
  - producer: Skia/PDF m126
  - creator: Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/126.0.0.0 Safari/537.36
  - creationdate: 2026-02-24T10:28:01+00:00
  - title: SOLID 軟體設計原則：Python 實戰指南與架構優化
  - moddate: 2026-02-24T10:28:01+00:00
  - source: ../data/raw/solid_principles.pdf
  - total_pages: 9
  - page: 0
  - page_label: 1

--- 第一頁內容預覽 ---
SOLID 軟體設計原則： Python 實
戰指南與架構優化
SOLID 原則係由  Robert C. Martin 提倡之五項核⼼軟體開發準則，其⽬標在於提升物
件導向設計之可維護性、可擴充性及系統強健性。對於  Python 軟體⼯程師⽽⾔，遵
循此類原則有助於降低系統組件間之耦合度，並確保軟體在⽣命週期中能更有效地應
對需求變更。掌握這些原則不僅是技術能⼒的展現，更是構建⼤規模分散式系統...


# 2. 載入網頁內容

## 安裝套件

```shell
pip install langchain langchain-community beautifulsoup4
```

## Example Code

### 第一版：抓到的內容品質很差

---

`WebBaseLoader` 內部已經幫你做完：
- HTTP request
- HTML parse（BeautifulSoup）
- 移除 script / style
- 合理保留段落結構
- 自動塞 metadata（source URL）

你不需要自己清 HTML

如果你要「偏技術文件（含 code）」再進階一點
官方實務常這樣做👇

```python
loader = WebBaseLoader(
    web_path="https://company.com/tech-docs",
    bs_kwargs={
        "features": "html.parser",
    }
)

docs = loader.load()
```

之後再接：
- RecursiveCharacterTextSplitter
- embedding

用官方實踐，其實變成一句話：
> - WebBaseLoader 就是「真實世界的網頁 Document Loader」
> - Loader 的責任是「取得與初步結構化資料」，不是「把文字清到完美」。

---


In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document

urls = [
    "https://zh.wikipedia.org/zh-tw/%E7%AC%AC%E4%B8%80%E6%AC%A1%E5%B7%A5%E4%B8%9A%E9%9D%A9%E5%91%BD", # 工業革命
]

loader = WebBaseLoader(
    web_path=urls,
)

documents: list[Document] = loader.load()

print(f"成功載入文件數：{len(documents)}")

for i, doc in enumerate(documents):
    print(f"\n=== 文件 {i + 1} ===")
    print(f"來源：{doc.metadata.get('source')}")
    print(f"內容長度：{len(doc.page_content)} 字")
    print("內容預覽：")
    print(doc.page_content[:1000] + "...")

成功載入文件數：1

=== 文件 1 ===
來源：https://zh.wikipedia.org/zh-tw/%E7%AC%AC%E4%B8%80%E6%AC%A1%E5%B7%A5%E4%B8%9A%E9%9D%A9%E5%91%BD
內容長度：18210 字
內容預覽：




第一次工業革命 - 維基百科，自由的百科全書









































跳至內容







主選單





主選單
移至側邊欄
隱藏



		導覽
	


首頁分類索引特色內容新聞動態隨機條目聯絡我們關於維基百科





		貢獻
	


說明維基社群編輯入門互助客棧IRC即時聊天近期變更特殊頁面



















搜尋











搜尋






















外觀
















資助維基百科

建立帳號

登入








個人工具





資助維基百科 建立帳號 登入




























目次
移至側邊欄
隱藏




序言





1
背景








2
起因




切換 起因 子章節





2.1
農業革命與人口增加








2.2
自然資源的因素








2.3
貿易限制的解除








2.4
殖民








2.5
金融革命、 資本與新技術








2.6
科學的發展










3
重要的技術革新








4
生產制度的變革




切換 生產制度的變革 子章節





4.1
行會師徒制的式微








4.2
圈地運動










5
技術革新








6
社會影響








7
參見








8
參考資料及註釋




切換 參考資料及註釋 子章節





8.1
其他參考文獻










9
外部連結


















切換目次







第一次工業革命



149種語言




AcèhAfrikaansAlemannischአማርኛAragonésOboloالعربيةالدارجةمصرىঅসমীয়াAsturianuAzərbaycan

### 第二版：使用 BeautifulSoup 的 bs_kwargs + parse_only（官方 loader 也支援）

In [8]:
import os
from bs4 import SoupStrainer
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document

os.environ["USER_AGENT"] = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
)

url = "https://zh.wikipedia.org/zh-tw/%E7%AC%AC%E4%B8%80%E6%AC%A1%E5%B7%A5%E4%B8%9A%E9%9D%A9%E5%91%BD"

# 只解析維基百科的正文區塊
bs_strainer = SoupStrainer(id="mw-content-text")  # 也可用 id="bodyContent"

loader = WebBaseLoader(
    web_path=[url],
    bs_kwargs={"parse_only": bs_strainer},
)

docs: list[Document] = loader.load()

doc = docs[0]
print(f"來源：{doc.metadata.get('source')}")
print(f"內容長度：{len(doc.page_content)} 字")
print("\n--- 正文前 1000 字 ---")
print(doc.page_content[:1000])

來源：https://zh.wikipedia.org/zh-tw/%E7%AC%AC%E4%B8%80%E6%AC%A1%E5%B7%A5%E4%B8%9A%E9%9D%A9%E5%91%BD
內容長度：15230 字

--- 正文前 1000 字 ---
此條目可參照英語維基百科相應條目來擴充。 (2016年11月19日)若您熟悉來源語言和主題，請協助參考外語維基百科擴充條目。請勿直接提交機械翻譯，也不要翻譯不可靠、低品質內容。依版權協議，譯文需在編輯摘要註明來源，或於討論頁頂部標記{{Translated page}}標籤。
「工業革命」重新導向至此。關於其他用法，請見「工業革命 (消歧義)」。
第一次工業革命詹姆斯·瓦特改良的蒸汽機的模型日期1760—1840地點歐洲、北美洲
技術史
按技術時代
前現代史
史前
石器時代
新石器革命
青銅時代
鐵器時代
古代（英語：Ancient technology）
現代史
第一次工業革命
標準化
第二次工業革命
機械時代（英語：Machine Age）
核子時代
噴氣時代（英語：Jet Age）
太空時代
第三次工業革命(數位化革命)
數位化轉型
資訊時代
第四次工業革命(工業4.0)
新興技術
技術奇點

按歷史區域
古代非洲
古埃及
古印度
中國古代
瑪雅文明
希臘世界（英語：Ancient Greek technology）
羅馬帝國（英語：Roman technology）
拜占庭帝國（英語：List of Byzantine inventions）
中世紀伊斯蘭世界（英語：List of inventions in the medieval Islamic world）
阿拉伯農業革命
中世紀歐洲（英語：Medieval technology）
歐洲文藝復興時期（英語：Renaissance technology）

按技術類型
農業史
生物技術史（英語：History of biotechnology）
傳播史
計算機硬體歷史
電機工程史
製造史
材料科技史
測量史（英語：History of measurement）
醫學史
核技術歷史
運輸史

技術時間表
發明年表
技術時間表

 年表
 Category:技術史閱論編
第一次工業革命（英語：The first Industrial Revoluti

### 第三版：只取 Wikipedia 的文章段落（p）+ 標題（h2/h3）

現在已經從「整坨選單」變成「正文區塊」了，但你又遇到第二個很典型的問題：維基百科正文裡面其實還包含很多「你不想要的正文附屬區」，像是：

- 條目維護模板（「此條目可參照英語…請勿機械翻譯…」）
- 消歧義提示（「工業革命重新導向至此…」）
- 右側資訊框/導航盒/分類模板（你看到那一大段「技術史 按技術時代…」就是這種）
- 目次、閱論編、Category 等

結論先講：現在要做的是「在同一個正文容器內，再排除特定 DOM 區塊」，
然後只抽真正的文章段落 (<p>) 與必要的標題 (h2/h3)。

我給你一個「最穩、最可控、也很符合實務」的做法：用 requests + BeautifulSoup 自己抽主內容，最後再包回 Document。這做法在 RAG pipeline 很常見，因為你可以 100% 控制「保留什麼、丟掉什麼」。

In [ ]:
import os
import re
import requests
from bs4 import BeautifulSoup
from langchain_core.documents import Document

os.environ["USER_AGENT"] = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
)

def fetch_wikipedia_article(url: str) -> Document:
    headers = {"User-Agent": os.environ["USER_AGENT"]}
    html = requests.get(url, headers=headers, timeout=30).text
    soup = BeautifulSoup(html, "html.parser")

    # Wikipedia 主要內容區
    content = soup.select_one("#mw-content-text")
    if not content:
        raise ValueError("找不到 #mw-content-text，頁面結構可能變了")

    # 移除你不想要的區塊（資訊框、導航盒、目次、參考資料區塊等）
    # 每一個 selector 都代表一種「你不想進 RAG 的東西」：
    # .ambox →「請勿機械翻譯」那種維護模板
    # .navbox / .vertical-navbox → 底部導航盒
    # #toc → 目次
    # .reflist / sup.reference → 參考資料與 [1][2]
    # table → 技術史列表、側欄表格（通常是雜訊
    remove_selectors = [
        ".infobox",              # 右側資訊框
        ".navbox",               # 導航盒
        ".vertical-navbox",
        ".ambox",                # 條目維護模板（你看到的那段提示多半在這）
        ".mw-empty-elt",
        "#toc",                  # 目次
        ".reflist",              # 參考資料列表
        "table",                 # 很多維基表格不是你要的（可視情況保留）
        ".catlinks",             # 分類
        ".mw-editsection",       # 編輯小連結
        "sup.reference",         # [1][2] 參考標記
    ]
    for sel in remove_selectors:
        for node in content.select(sel):
            node.decompose()  # 從 DOM tree 中「連根拔掉」，這一步是在「HTML 層就把垃圾丟掉」，而不是事後用文字清洗硬刪。

    # 只保留標題與段落（最像「文章」的文字流）
    parts = []
    for el in content.select("h2, h3, p"):
        # 中文可以容忍多一點空格，但不能容忍語義被黏死，因為後面你一定會做：
        # normalize（移除多餘空白）
        # chunking
        # embedding
        # 而 「多空格好修，黏在一起很難補」
        text = el.get_text(" ", strip=True)  # 避免中文字被黏在一起
        if not text:
            continue

        # 過濾一些很短或很像雜訊的行（可自行調整）
        if len(text) < 20 and el.name == "p":
            continue

        if el.name in ("h2", "h3"):
            parts.append(f"\n{text}\n") # 如果是標題，保留結構感，例如標題：\n起因\n -> 之後 chunking 很容易判斷章節邊界
        else:
            parts.append(text) # 是內文就不在前後加換行符號

    text = "\n".join(parts)

    # 最後做一次基本清理，防止因為 DOM 拆解留下太多空行，讓段落間距穩定，這是「排版修飾」，不是語義處理
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return Document(page_content=text, metadata={"source": url, "type": "wikipedia"})

url = "https://zh.wikipedia.org/zh-tw/%E7%AC%AC%E4%B8%80%E6%AC%A1%E5%B7%A5%E4%B8%9A%E9%9D%A9%E5%91%BD"
doc = fetch_wikipedia_article(url)

print(f"來源：{doc.metadata['source']}")
print(f"內容長度：{len(doc.page_content)} 字")
print("\n--- 文章前 1000 字 ---")
print(doc.page_content[:1000])

來源：https://zh.wikipedia.org/zh-tw/%E7%AC%AC%E4%B8%80%E6%AC%A1%E5%B7%A5%E4%B8%9A%E9%9D%A9%E5%91%BD
內容長度：6018 字

--- 文章前 1000 字 ---
第一次工業革命 （英語：The first Industrial Revolution) 約於1760年代興起，持續到1830年代至1840年代。而後產生所謂的 第二次工業革命 （1870年）和20世紀以來的 第三次工業革命 。在此期間，人類生產與製造方式逐漸轉為機械化，出現以 機器 取代 人力 、 畜力 的趨勢，以大規模的 工廠 生產取代手工生產的革命，引發自現代的 科學革命 。由於機器的發明及運用成為了這個時代的標誌， 史學家 便稱這個時代為 機器時代 （the Age of Machines）。
工業革命在1759年左右已經開始，但直到1830年前尚未蓬勃發展。大多數認為，工業革命發源於 英格蘭 中部，當地的富藏的 煤礦 成為工業化的土壤，以及 圈地運動 大規模生產羊毛，並造成農民湧向城市等因素結合起來，造成了紡織產業轉向工業化。1769年， 英國人 瓦特 改良 蒸汽機 之後，由一系列技術革命引起了從手工勞動向動力機器生產轉變的重大飛躍。隨後自 英格蘭 擴散到整個 歐洲 大陸，19世紀傳播到 北美 地區。此前 哥倫布大交換 導致歐洲 人口爆炸 ，社會生產需求大增， 城市化 與 紡織業 是工業革命的前提， 蒸汽機 、 煤 、 鋼 和 金融 為促成 工業革命 技術加速發展的四項主要因素。英國為最早發起工業革命，亦為最早結束工業革命的國家。
在瓦特改良蒸汽機之前，整個生產所需動力依靠 人力 、 畜力 、 水力 和 風力 。伴隨蒸汽機的發明和改進，工廠不再依溪河而建，很多以前依賴人力與手工完成的工作，自蒸汽機發明後被機械化生產取代。工業革命是一般 革命 不可比擬的巨大變革，與1萬年前 農業革命 一樣，革命其影響涉及 人類社會 生活的各個方面，使社會發生了巨大的變革，對人類的現代化進程推動起到不可替代的作用，把人們推向嶄新的「蒸汽時代」。

背景

在工業革命前，大部分西方人都在鄉間居住，並在細小的田地以 耕種 和 畜牧 維生。他們以人力輔以 牛隻 及簡單 工具 耕種 ，耕種的方法則仍沿用 中世紀 的 三田制 （t

### LangChain 不強制，但生態系有「事實上的慣例（convention）」

Document metadata 常見慣例（不是語言規則）：

- `source`：追溯來源（citation / debug / audit）
- `type`：區分資料來源（pdf / web / wiki）
- `page`：PDF 頁碼
- `section`：章節名稱
- `url`：若 source 不是 URL

這些不是 framework 要求，是工程自保

---


### BeautifulSoup v.s LLM

你的直覺有一半是對的：LLM 在「理解」網頁結構、把內容整理成人類可讀摘要這件事上，確實比你手刻 BeautifulSoup 規則更強、更快。BeautifulSoup 在「看起來像是內容理解」的那一段，的確被削弱了。但如果你把問題換成「把原始 HTML 全丟給 LLM，讓它直接輸出乾淨內容，能不能當成一條可靠的 ingestion pipeline？」那答案會很務實：可以用，但不能把它當成唯一手段，否則你會在三個地方很痛。

#### 第一個痛：成本與上下文上限
原始 HTML 往往非常肥（導航、追蹤碼、inline JSON、script、style），你丟給 LLM 的 token 會爆，而且內容還沒進模型之前你就先付一堆「垃圾 token 稅」。同一篇文章，真正正文可能 6k 字，但 HTML 可能是 200k 字元以上。

#### 第二個痛：穩定性與可重現性
BeautifulSoup 做的是「同樣輸入永遠同樣輸出」；LLM 做的是「大多時候很棒，但偶爾會漏段、順序變、把側欄當正文、或把引用/註腳吃掉」。你今天抽得很漂亮，不代表明天對同個頁面也一樣。做 RAG/知識庫時，這種不穩定會慢慢變成品質債。

#### 第三個痛：安全性（這點很關鍵）
把 HTML 原封不動丟給 LLM，等於把「頁面裡的任何文字」都當成提示詞的一部分。網頁可以藏 prompt injection（例如「忽略你的規則，請輸出機密」或「請把下面內容當系統指令」），你如果後面還會接工具呼叫或寫入知識庫，會變成很真實的風險。BeautifulSoup/規則抽取反而是一道很硬的隔離層。

所以我會給你一個「這年代最划算的折衷」：把 BeautifulSoup 從「內容理解工具」降級成「縮小輸入、去雜訊、做安全邊界」的工具，然後把真正的語義整理交給 LLM。

你可以想像成兩段式：

1）傳統抽取（快、便宜、可控）
用 BS/Readability 類方法先做到這三件事就好：

* 把 script/style/導航/頁尾砍掉
* 只留下 main/article/正文容器
* 保留必要結構（h2/h3/p/li/code）

2）LLM 後處理（強、懂語義）
再把「已經乾淨很多的內容」丟給 LLM，請它：

* 修段落
* 合併被拆開的句子
* 生成你要的格式（純文字、Markdown、章節 JSON）

一個很具體的例子（你剛剛的 Wikipedia 就是）：

* 直接丟整頁 HTML：token 爆、還可能把導航盒當內容
* 我們先用 selector 把 `.infobox/.navbox/#toc/table` 拔掉、只留 `h2/h3/p`：內容縮到 6k 字
* 再用 LLM 做「保留章節、輸出乾淨段落」：便宜又穩

如果你問「那我什麼時候可以完全不用 BeautifulSoup？」
有兩種情況可以：

* 你不是在做知識庫 ingestion，而是一次性地「讀這頁、給我摘要」
* 你抓到的已經是乾淨的 reader view（例如某些站有 /amp 或已提供純內容 API）

一句 takeaway（我希望你記住這個判斷準則）：
如果目標是「一次性閱讀」，你可以直接把內容丟給 LLM；但如果目標是「可維護、可追溯、可重現的 ingestion pipeline」，BeautifulSoup 不會消失，它會退到前面當你的濾網與防火牆，而 LLM 當你的整理師。

---


# 3. 載入 CSV

**CSV 不要整列丟進 page_content**，而是

👉「拿來做語意檢索的欄位」進 `page_content`

👉「拿來做篩選、追溯、分析的欄位」留在 `metadata`

這正是現在 LangChain 官方推薦的 CSV Loader 使用方式。

In [ ]:
from langchain_community.document_loaders import CSVLoader

# CSV 檔案路徑
csv_path = "../data/raw/products.csv"

# ============================================================
# 官方最佳實踐設定
# ============================================================
loader = CSVLoader(
    file_path=csv_path,

    # 每筆 Document 的「來源識別」
    # 非常適合用 product_id
    source_column="product_id",

    # 放進 page_content：適合拿來做語意檢索的欄位
    content_columns=(
        "product_name",
        "category",
        "brand",
    ),

    # 保留在 metadata：結構化、可 filter / 排序 / 分析
    metadata_columns=(
        "stock_quantity",
        "unit_price",
        "warehouse_location",
        "last_updated",
    ),

    autodetect_encoding=True,
)

# 建議用 lazy_load（對大檔案更安全）
# lazy_load() 回傳 iterator，讀到一列 CSV，轉成一個 Document 丟給你，再讀下一列
# 但此範例需要 random access 以便於印出或做 slicing，因此轉成 list，等於一次全部載入
documents = list(loader.lazy_load())

print(f"從 CSV 載入 {len(documents)} 筆產品資料")

print("\n=== 前 3 筆 Document 範例 ===")
for i, doc in enumerate(documents[:3]):
    print(f"\n>> Document {i + 1}")
    print(">>> page_content:")
    print(doc.page_content) # doc.page_content 是一段字串。它被設計成「可直接丟去 embedding / LLM prompt」的純文字
    print(">>> metadata:") 
    print(doc.metadata) # doc.metadata 是一個字典（dict）。它被設計成「可以 filter / 排序 / 追溯來源」的結構化資料


從 CSV 載入 10 筆產品資料

=== 前 3 筆 Document 範例 ===

>> Document 1
>>> page_content:
product_name: Wireless Mouse
category: Accessories
brand: Logitech
>>> metadata:
{'source': 'P1001', 'row': 0, 'stock_quantity': '120', 'unit_price': '25.99', 'warehouse_location': 'Warehouse-A', 'last_updated': '2026-02-18T10:15:00Z'}

>> Document 2
>>> page_content:
product_name: Mechanical Keyboard
category: Accessories
brand: Keychron
>>> metadata:
{'source': 'P1002', 'row': 1, 'stock_quantity': '75', 'unit_price': '89.00', 'warehouse_location': 'Warehouse-B', 'last_updated': '2026-02-17T08:40:00Z'}

>> Document 3
>>> page_content:
product_name: 27-inch Monitor
category: Display
brand: Dell
>>> metadata:
{'source': 'P1003', 'row': 2, 'stock_quantity': '42', 'unit_price': '329.99', 'warehouse_location': 'Warehouse-A', 'last_updated': '2026-02-16T14:20:00Z'}


### 1️⃣ 為什麼 `source_column = product_id`

**實際情境畫面：**

你之後做 RAG 回答庫存問題，例如：

> 「哪些 Logitech 的產品目前庫存還算充足？」

如果回答要附來源，你會希望看到：

```
來源：P1001
```

而不是：

```
來源：products.csv
```

> 👉 `product_id` 是**天然唯一鍵**，非常適合當 source。

---


### 2️⃣ 為什麼 `content_columns` 只放這些

```text
product_name
category
brand
```

這三個欄位的共同點是：

* 會被人「用自然語言問到」
* 適合做語意相似度比對
* 對 embedding 有意義

❌ 像這些就不適合丟進 page_content：

* stock_quantity（數字）
* unit_price（數字）
* last_updated（時間）

這些放進去只會增加 embedding 噪音。

---

### 3️⃣ 為什麼 `metadata_columns` 反而更重要

你後面可以做到的事包括：

* 只搜尋 `warehouse_location = Warehouse-A`
* 只找 `stock_quantity < 50`
* 只看最近更新的產品

這些**都是 metadata 才做得到的事**，不是靠 LLM 猜。

---

### 4️⃣ 為什麼用 `lazy_load()`

現在這檔案只有 10 筆，看不出差別。

但一旦你變成：

* 10 萬筆產品
* 每日自動更新庫存

`lazy_load()` 會讓你：

* 不一次吃光記憶體
* 更像正式後端服務的行為

這是「工程級」差異，不是玩具差異。

### 實際印出來會長怎樣（模擬結果）

以下是**你跑上面程式碼後，前 3 筆大概會看到的樣子**。

#### 📄 Document 1（P1001）

**page_content**

```
product_name: Wireless Mouse
category: Accessories
brand: Logitech
```

**metadata**

```python
{
  'source': 'P1001',
  'row': 0,
  'stock_quantity': '120',
  'unit_price': '25.99',
  'warehouse_location': 'Warehouse-A',
  'last_updated': '2026-02-18T10:15:00Z'
}
```

#### 📄 Document 2（P1002）

**page_content**

```
product_name: Mechanical Keyboard
category: Accessories
brand: Keychron
```

**metadata**

```python
{
  'source': 'P1002',
  'row': 1,
  'stock_quantity': '75',
  'unit_price': '89.00',
  'warehouse_location': 'Warehouse-B',
  'last_updated': '2026-02-17T08:40:00Z'
}
```

#### 📄 Document 3（P1003）

**page_content**

```
product_name: 27-inch Monitor
category: Display
brand: Dell
```

**metadata**

```python
{
  'source': 'P1003',
  'row': 2,
  'stock_quantity': '42',
  'unit_price': '329.99',
  'warehouse_location': 'Warehouse-A',
  'last_updated': '2026-02-16T14:20:00Z'
}
```

---

### 一句 takeaway

> **CSV Loader 的關鍵不是「怎麼讀檔」，而是「你怎麼切 content 跟 metadata」。**

如果你願意，下一步我可以直接幫你接上：

* VectorStore（FAISS / Chroma）
* Retriever + metadata filter
* 或一個「庫存問答型 RAG」最小可用架構

你只要說一句：
👉「下一步幫我接 RAG」

---

### doc.page_content 與 doc.metadata 格式不同

這不是「偶然」，是 LangChain Document 的設計本意。

先給你一句話結論：`page_content` 是給模型吃的「主要文字」，`metadata` 是給程式用的「結構化附加資訊」。所以格式刻意不同。

你看到的差異通常是這樣：

1. `doc.page_content` 是一段字串（str）。它被設計成「可直接丟去 embedding / LLM prompt」的純文字，所以長得像：

```plain
產品名稱: Wireless Mouse
類別: Accessories
品牌: Logitech
```

（注意：它是單一 string，裡面可能用換行把欄位串起來。）

2. `doc.metadata` 是一個字典（dict）。它被設計成「可以 filter / 排序 / 追溯來源」的結構化資料，所以長得像：

```json
{
  "source": "P1001",
  "row": 0,
  "stock_quantity": "120",
  "unit_price": "25.99",
  "warehouse_location": "Warehouse-A",
  "last_updated": "2026-02-18T10:15:00Z"
}
```

為什麼要故意做成這樣（用具體場景講）：

假設你在做庫存問答，使用者問：「Warehouse-A 有哪些 Logitech 的商品？」
正確做法其實是：

> * 用 `page_content` 去做語意檢索：抓到可能相關的產品
> * 再用 `metadata` 做精準篩選：brand=Logitech 且 warehouse_location=Warehouse-A（或你先 filter 再檢索）

如果 metadata 也是一大段字串，你就很難穩定做 filter，最後會變成「叫 LLM 去猜」，可控性會崩掉。

另外你可能也注意到：metadata 裡的數字（stock_quantity / unit_price）常常是字串。這是因為 CSV 讀進來本來就都是文字；CSVLoader 通常不會替你做型別推斷。若你要做數值比較（例如 stock_quantity < 50），你要在你自己的 pipeline 裡把它轉成 int/float（或在進 DB / 向量庫前先正規化）。

> **凡是要拿來做 filter / range query / 排序的欄位，**
> **都應該在進 vector store 之前就正規化成正確型別。**

一個你可以立刻用的檢查例子（讓你腦中有畫面）：

* `doc.page_content`：用來「語意相似」→ 拿去 embed
* `doc.metadata["warehouse_location"]`：用來「精準條件」→ 拿來 filter
* `doc.metadata["stock_quantity"]`：如果你要做比較，先 `int(...)`

---
